In [ ]:
# 步骤1：安装依赖库（首次运行时执行，已安装则跳过）
!pip install pandas numpy matplotlib scikit-learn xgboost joblib --upgrade

In [ ]:
# 步骤2：导入所需库 + 全局中文字体设置（核心修改：解决中文乱码）
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer  # 处理缺失值
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, roc_curve, confusion_matrix)
from sklearn.decomposition import PCA  # 无监督可视化用
from xgboost import XGBClassifier
import joblib
import warnings
warnings.filterwarnings('ignore')  # 忽略无关警告

# --------------------------
# 关键：设置matplotlib中文字体（兼容多系统）
# --------------------------
plt.rcParams['font.sans-serif'] = [
    'WenQuanYi Zen Hei',  # Linux默认中文字体
    'SimHei',             # Windows默认中文字体
    'Arial Unicode MS',   # Mac默认中文字体
    'DejaVu Sans'         # 备选字体（防止上述字体缺失）
]
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示异常问题
%matplotlib inline  # 确保图表在Notebook中正常显示

In [ ]:
# 步骤3：模拟客户流失数据集
def generate_churn_data(n_samples=10000):
    """
    生成客户流失数据集
    参数：n_samples - 数据总行数
    返回：df - 包含特征和流失标签的DataFrame
    """
    np.random.seed(42)  # 固定随机种子，确保结果可复现
    
    # 1. 数值型特征（客户属性+行为数据）
    data = {
        "customer_age": np.random.normal(35, 10, n_samples).astype(int).clip(18, 65),  # 年龄（18-65岁）
        "monthly_spend": np.random.normal(80, 30, n_samples).clip(20, 200),  # 月消费（20-200元）
        "service_usage": np.random.poisson(15, n_samples).clip(1, 30),  # 月服务使用次数（1-30次）
        "support_calls": np.random.poisson(2, n_samples).clip(0, 10),  # 月客服呼叫次数（0-10次）
        "tenure": np.random.poisson(18, n_samples).clip(1, 60)  # 客户在网时长（1-60个月）
    }
    
    # 2. 分类型特征（客户偏好+服务类型）
    data["gender"] = np.random.choice(["Male", "Female"], n_samples, p=[0.52, 0.48])
    data["payment_method"] = np.random.choice(["Credit Card", "Bank Transfer", "Cash"], n_samples, p=[0.6, 0.3, 0.1])
    data["service_type"] = np.random.choice(["Basic", "Premium", "Enterprise"], n_samples, p=[0.5, 0.3, 0.2])
    data["contract_duration"] = np.random.choice([1, 12, 24], n_samples, p=[0.4, 0.3, 0.3])  # 合约期限（1=月付，12=年付）
    
    # 3. 生成流失标签（基于业务逻辑：高风险行为→高流失概率）
    churn_prob = (
        0.3 * (data["support_calls"] >= 3) +  # 客服呼叫≥3次：流失概率+30%
        0.4 * (data["contract_duration"] == 1) +  # 月付合约：流失概率+40%
        0.2 * (data["tenure"] <= 6) +  # 在网≤6个月：流失概率+20%
        0.1 * (data["monthly_spend"] < 50)  # 月消费<50元：流失概率+10%
    )
    churn_prob += np.random.normal(0, 0.1, n_samples)  # 加入随机噪声
    churn_prob = np.clip(churn_prob, 0, 1)  # 概率限制在0-1之间
    data["churn"] = np.random.binomial(1, churn_prob).astype(int)  # 0=留存，1=流失
    
    # 4. 加入少量缺失值（模拟真实数据）
    df = pd.DataFrame(data)
    df.loc[np.random.choice(df.index, int(n_samples*0.05)), "monthly_spend"] = np.nan  # 5%缺失
    df.loc[np.random.choice(df.index, int(n_samples*0.03)), "support_calls"] = np.nan  # 3%缺失
    
    return df

# 生成数据并查看基本信息
df = generate_churn_data(n_samples=10000)
print(f"数据形状：{df.shape}（行：客户数，列：特征数）")
print(f"整体流失率：{df['churn'].mean():.2%}（行业常见范围：5%-20%）")
print("\n数据前5行：")
# print(df.head())
print("\n数据缺失值统计：")
print(df.isnull().sum())
df.head()

In [ ]:
# 步骤4：创建数据预处理流水线
def create_preprocessor():
    """
    构建数据预处理流水线
    返回：
        preprocessor - 预处理流水线（ColumnTransformer）
        numerical_features - 数值型特征列表
        categorical_features - 分类型特征列表
    """
    # 1. 划分特征类型
    numerical_features = ["customer_age", "monthly_spend", "service_usage", "support_calls", "tenure"]
    categorical_features = ["gender", "payment_method", "service_type", "contract_duration"]
    
    # 2. 数值型特征处理器
    numerical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="mean")),  # 缺失值用均值填充
        ("scaler", StandardScaler())  # 标准化（均值=0，标准差=1）
    ])
    
    # 3. 分类型特征处理器
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),  # 缺失值用众数填充
        ("onehot", OneHotEncoder(handle_unknown="ignore"))  # 独热编码（忽略未见过的类别）
    ])
    
    # 4. 合并处理器（对不同特征应用不同处理）
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numerical_transformer, numerical_features),
            ("cat", categorical_transformer, categorical_features)
        ])
    
    return preprocessor, numerical_features, categorical_features

# 测试预处理流水线（查看特征维度变化）
preprocessor, num_feat, cat_feat = create_preprocessor()
X_sample = df.drop("churn", axis=1)  # 特征数据
X_processed = preprocessor.fit_transform(X_sample)  # 预处理后的数据
print(f"原始特征数：{X_sample.shape[1]}")
print(f"预处理后特征数：{X_processed.shape[1]}（分类型特征编码后维度增加）")

In [ ]:
X_sample.head()

In [ ]:
print(X_processed)

In [ ]:
# 步骤5：训练有监督学习模型
def train_supervised_models(X, y):
    """
    训练有监督模型并评估
    参数：
        X - 特征数据
        y - 目标变量（churn：0=留存，1=流失）
    返回：
        best_model - 最优模型（AUC-ROC最高）
        model_performance - 所有模型性能指标
        X_test, y_test - 测试集数据
    """
    # 1. 划分训练集/测试集（8:2，分层抽样确保流失率一致）
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"训练集规模：{X_train.shape[0]} 客户，测试集规模：{X_test.shape[0]} 客户")
    
    # 2. 获取预处理流水线
    preprocessor, _, _ = create_preprocessor()
    
    # 3. 定义模型与超参数搜索空间
    models = {
        "Logistic Regression": LogisticRegression(
            class_weight="balanced", max_iter=1000, random_state=42
        ),
        "Random Forest": RandomForestClassifier(
            class_weight="balanced", random_state=42
        ),
        "XGBoost": XGBClassifier(
            scale_pos_weight=sum(y==0)/sum(y==1),  # 处理类别不平衡
            random_state=42, use_label_encoder=False, eval_metric="logloss"
        )
    }
    
    param_grids = {
        "Logistic Regression": {"classifier__C": [0.01, 0.1, 1, 10]},  # 正则化强度
        "Random Forest": {
            "classifier__n_estimators": [100, 200],  # 树数量
            "classifier__max_depth": [5, 10, None]  # 树深度
        },
        "XGBoost": {
            "classifier__n_estimators": [100, 200],
            "classifier__learning_rate": [0.01, 0.1],  # 学习率
            "classifier__max_depth": [3, 5]
        }
    }
    
    # 4. 训练与评估模型
    model_performance = {}
    best_model = None
    best_auc = 0
    
    for name, model in models.items():
        print(f"\n=== 训练 {name} ===")
        # 构建完整流水线（预处理+模型）
        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("classifier", model)
        ])
        
        # 网格搜索调参（5折交叉验证，优化AUC-ROC）
        grid_search = GridSearchCV(
            pipeline, param_grids[name], cv=5, scoring="roc_auc", n_jobs=-1
        )
        grid_search.fit(X_train, y_train)
        
        # 预测与评估
        y_pred = grid_search.predict(X_test)
        y_pred_proba = grid_search.predict_proba(X_test)[:, 1]  # 流失概率
        
        # 计算关键指标
        metrics = {
            "Accuracy": round(accuracy_score(y_test, y_pred), 3),  # 准确率
            "Precision": round(precision_score(y_test, y_pred), 3),  # 精确率（预测流失的准确性）
            "Recall": round(recall_score(y_test, y_pred), 3),  # 召回率（漏判率，越低越好）
            "F1-Score": round(f1_score(y_test, y_pred), 3),  # 精确率+召回率平衡
            "AUC-ROC": round(roc_auc_score(y_test, y_pred_proba), 3)  # 核心指标（越高越好）
        }
        model_performance[name] = metrics
        
        # 输出结果
        print(f"最优参数：{grid_search.best_params_}")
        print(f"测试集性能：{metrics}")
        
        # 更新最优模型
        if metrics["AUC-ROC"] > best_auc:
            best_auc = metrics["AUC-ROC"]
            best_model = grid_search.best_estimator_
    
    # 5. 输出模型对比
    print("\n=== 模型性能对比（按AUC-ROC排序）===")
    perf_df = pd.DataFrame(model_performance).T.sort_values("AUC-ROC", ascending=False)
    print(perf_df)
    
    return best_model, model_performance, X_test, y_test

# 准备特征与目标变量
X = df.drop("churn", axis=1)
y = df["churn"]

# 训练有监督模型
best_supervised_model, _, X_test, y_test = train_supervised_models(X, y)
print(f"\n最优模型：{best_supervised_model.named_steps['classifier'].__class__.__name__}")

In [ ]:
# 步骤6：可视化工具函数
def plot_roc_curve(model, X_test, y_test):
    """绘制ROC曲线（评估模型区分能力）"""
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    auc = round(roc_auc_score(y_test, y_pred_proba), 3)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"最优模型 (AUC = {auc})")
    plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")  # 随机猜测线
    plt.xlabel("假正例率（FPR）：误判为流失的正常客户比例")
    plt.ylabel("真正例率（TPR）：正确识别的流失客户比例")
    plt.title("客户流失预测 ROC 曲线（AUC越高，模型越好）")
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.show()

def plot_feature_importance(model):
    """绘制特征重要性（仅支持树模型：随机森林/XGBoost）"""
    # 1. 获取特征名称（数值型+独热编码后的分类型）
    preprocessor = model.named_steps["preprocessor"]
    num_feat = ["customer_age", "monthly_spend", "service_usage", "support_calls", "tenure"]
    cat_ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
    cat_feat = cat_ohe.get_feature_names_out(["gender", "payment_method", "service_type", "contract_duration"])
    all_feat = list(num_feat) + list(cat_feat)
    
    # 2. 获取特征重要性
    classifier = model.named_steps["classifier"]
    if hasattr(classifier, "feature_importances_"):
        importances = classifier.feature_importances_
    else:
        print("该模型不支持特征重要性分析（仅树模型支持）")
        return
    
    # 3. 排序并可视化前10个重要特征
    feat_importance = pd.DataFrame({
        "feature": all_feat,
        "importance": importances
    }).sort_values("importance", ascending=False).head(10)
    
    plt.figure(figsize=(10, 6))
    plt.barh(feat_importance["feature"], feat_importance["importance"], color="teal")
    plt.xlabel("特征重要性（值越高，对流失预测影响越大）")
    plt.ylabel("特征名称")
    plt.title("客户流失预测 - 前10个关键特征")
    plt.gca().invert_yaxis()  # 倒序：重要性高的在顶部
    plt.grid(alpha=0.3, axis="x")
    plt.show()

# 可视化最优模型
print("=== 最优模型 ROC 曲线 ===")
plot_roc_curve(best_supervised_model, X_test, y_test)

print("\n=== 最优模型 特征重要性 ===")
plot_feature_importance(best_supervised_model)

In [ ]:
# 步骤7：训练无监督学习模型（已修复X_pca未定义问题）
def train_unsupervised_models(df, y_true):
    """
    训练无监督模型：K-means分群 + 孤立森林异常检测
    返回：分群模型、异常检测模型、各聚类流失率统计
    """
    # 1. 数据预处理（仅用数值型特征）
    _, num_feat, _ = create_preprocessor()
    X_num = df[num_feat].copy()
    
    # 缺失值填充 + 标准化
    imputer = SimpleImputer(strategy="mean")
    scaler = StandardScaler()
    X_num_processed = scaler.fit_transform(imputer.fit_transform(X_num))
    
    # --------------------------
    # 2. K-means 客户分群
    # --------------------------
    print("=== K-means 客户分群 ===")
    # 肘部法则选最优K
    inertia = []
    for k in range(2, 7):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(X_num_processed)
        inertia.append(kmeans.inertia_)
    
    # 可视化肘部法则
    plt.figure(figsize=(8, 4))
    plt.plot(range(2, 7), inertia, marker="o", color="purple")
    plt.xlabel("聚类数量（K）")
    plt.ylabel("惯性（簇内平方和，越小表示簇内越集中）")
    plt.title("K-means 肘部法则（选择拐点处的K值）")
    plt.grid(alpha=0.3)
    plt.show()
    
    # 训练K-means（K=4）
    kmeans_model = KMeans(n_clusters=4, random_state=42, n_init=10)
    df["cluster"] = kmeans_model.fit_predict(X_num_processed)
    
    # 分析各聚类流失率
    cluster_churn = df.groupby("cluster")["churn"].agg([
        "count", "mean"
    ]).round(3)
    cluster_churn.columns = ["客户数量", "流失率"]
    cluster_churn["风险等级"] = cluster_churn["流失率"].apply(
        lambda x: "高风险" if x >= 0.3 else "中风险" if x >= 0.15 else "低风险"
    )
    print("\n各聚类客户流失率统计：")
    print(cluster_churn)
    
    # 可视化聚类流失率
    plt.figure(figsize=(8, 4))
    cluster_churn["流失率"].plot(kind="bar", color=["green", "yellow", "orange", "red"])
    plt.xlabel("客户聚类")
    plt.ylabel("流失率")
    plt.title("各聚类客户流失率对比（红色=高风险）")
    plt.xticks(rotation=0)
    plt.grid(alpha=0.3, axis="y")
    plt.show()
    
    # --------------------------
    # 3. 孤立森林 异常检测（已修复X_pca问题）
    # --------------------------
    print("\n=== 孤立森林 异常检测 ===")
    # 训练孤立森林
    iforest_model = IsolationForest(
        contamination=0.2,  # 异常比例（接近真实流失率）
        random_state=42
    )
    df["is_anomaly"] = iforest_model.fit_predict(X_num_processed)
    df["anomaly_label"] = df["is_anomaly"].map({-1: 1, 1: 0})  # 转换为1=潜在流失，0=正常
    
    # 评估异常检测效果
    accuracy = round(accuracy_score(y_true, df["anomaly_label"]), 3)
    recall = round(recall_score(y_true, df["anomaly_label"]), 3)
    print(f"异常检测准确率：{accuracy}")
    print(f"流失客户识别率（召回率）：{recall}")
    
    # --------------------------
    # 关键修复：添加PCA降维，定义X_pca变量
    # --------------------------
    pca = PCA(n_components=2)  # 初始化PCA，降维到2维（用于可视化）
    X_pca = pca.fit_transform(X_num_processed)  # 对预处理后的数值特征做PCA转换
    
    # 可视化异常检测结果（此时X_pca已定义，无报错）
    plt.figure(figsize=(8, 6))
    # 正常客户（蓝色）
    plt.scatter(
        X_pca[df["is_anomaly"]==1, 0], X_pca[df["is_anomaly"]==1, 1],
        c="blue", label="正常客户", alpha=0.5, s=50
    )
    # 潜在流失客户（红色）
    plt.scatter(
        X_pca[df["is_anomaly"]==-1, 0], X_pca[df["is_anomaly"]==-1, 1],
        c="red", label="潜在流失客户", alpha=0.7, s=50
    )
    plt.xlabel("PCA维度1")
    plt.ylabel("PCA维度2")
    plt.title("孤立森林异常检测结果（红色=需重点关注）")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
    
    return kmeans_model, iforest_model, cluster_churn

# 训练无监督模型（此时无X_pca未定义错误）
kmeans_model, iforest_model, cluster_churn = train_unsupervised_models(df.copy(), y)

In [ ]:
# 步骤8：模型保存与新客户预测（已修复数组长度不一致错误）
def save_model(model, save_path="best_churn_model.pkl"):
    """保存模型到本地"""
    joblib.dump(model, save_path)
    print(f"模型已保存至：{save_path}")

def load_model(load_path="best_churn_model.pkl"):
    """加载本地模型"""
    return joblib.load(load_path)

def predict_churn_risk(model, new_customer_data):
    """
    批量预测新客户流失风险（修复数组长度问题）
    参数：
        model - 训练好的模型
        new_customer_data - 新客户数据（DataFrame，n行客户）
    返回：
        result - 预测结果（DataFrame，n行，与客户数量一致）
    """
    # 1. 批量预测所有客户（去掉[0]，获取完整结果数组）
    churn_probs = model.predict_proba(new_customer_data)[:, 1]  # 形状：(n_samples,)
    churn_preds = model.predict(new_customer_data)              # 形状：(n_samples,)
    print(f"预测客户数量：{len(new_customer_data)}")
    print(f"流失概率数组长度：{len(churn_probs)}, 预测标签数组长度：{len(churn_preds)}（与客户数量一致）")
    
    # 2. 批量生成风险等级（列表推导式，确保长度一致）
    risk_levels = []
    for prob in churn_probs:
        if prob >= 0.6:
            risk_levels.append("高风险（需立即干预，如专属优惠）")
        elif prob >= 0.3:
            risk_levels.append("中风险（需重点关注，如满意度调查）")
        else:
            risk_levels.append("低风险（常规维护即可）")
    print(f"风险等级列表长度：{len(risk_levels)}（与客户数量一致）")
    
    # 3. 整理结果（所有列长度均为n_samples，无长度不一致错误）
    result = pd.DataFrame({
        "客户ID": new_customer_data.index,  # 长度：n_samples
        "流失预测": ["是" if pred == 1 else "否" for pred in churn_preds],  # 长度：n_samples
        "流失概率": [f"{prob:.2%}" for prob in churn_probs],  # 长度：n_samples
        "风险等级": risk_levels  # 长度：n_samples
    })
    
    return result

# 1. 保存最优模型
save_model(best_supervised_model)

# 2. 测试多客户预测（验证无长度不一致错误）
print("\n=== 新客户流失风险预测示例（3个客户）===")
new_customers = pd.DataFrame({
    "customer_age": [28, 45, 32],          # 3个客户
    "monthly_spend": [45.8, 120.5, 78.2],  # 3个客户
    "contract_duration": [1, 24, 12],      # 3个客户
    "service_usage": [8, 25, 15],          # 3个客户
    "support_calls": [5, 1, 2],            # 3个客户
    "tenure": [2, 36, 10],                 # 3个客户
    "gender": ["Female", "Male", "Female"],# 3个客户
    "payment_method": ["Cash", "Credit Card", "Bank Transfer"],  # 3个客户
    "service_type": ["Basic", "Enterprise", "Premium"]           # 3个客户
}, index=["CUST-2025001", "CUST-2025002", "CUST-2025003"])  # 3个客户ID

# 执行预测（此时无“数组长度不一致”错误）
pred_result = predict_churn_risk(best_supervised_model, new_customers)
print("\n预测结果：")
print(pred_result)